# Cylindrical Slicer for 3D Models

This notebook implements a slicer for cylindrical 3D printers that:
- Loads 3D models (STL files)
- Slices them using cylindrical surfaces (radial layers)
- Extracts contours from each slice
- Generates multi-wall toolpaths

## Coordinate System
- **X**: Linear position along rotation axis (mm)
- **Y**: Rotation angle (degrees)
- **Z**: Height above drum surface (mm)

## Import Libraries

In [ ]:
import numpy as np
import fullcontrol as fc
from fullcontrol.geometry.cylindrical import (
    cylindrical_to_point,
    cylindrical_to_cartesian,
    convert_steps_for_visualization
)
import trimesh
from shapely.geometry import Polygon, MultiPolygon, Point, LineString
from shapely.ops import unary_union
from scipy.spatial import ConvexHull
import matplotlib.pyplot as plt
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

## Slicer Configuration

In [ ]:
# Printer configuration
DRUM_DIAMETER = 56  # mm
DRUM_RADIUS = DRUM_DIAMETER / 2  # 28mm
LAYER_HEIGHT = 0.2  # mm (radial layer thickness)
EXTRUSION_WIDTH = 0.4  # mm
LINE_OVERLAP_PERCENT = 10  # % overlap between walls
FLOW_PERCENT = 100  # %

# Slicer settings
NUM_WALLS = 2  # Number of wall perimeters
WALL_SPACING = EXTRUSION_WIDTH * (1 - LINE_OVERLAP_PERCENT / 100)  # 0.36mm with overlap

# Infill settings
TOP_BOTTOM_LAYERS = 1  # Number of solid top/bottom layers
INFILL_LINE_SPACING = EXTRUSION_WIDTH * (1 - LINE_OVERLAP_PERCENT / 100)  # Same as wall spacing
INFILL_ANGLE_A = 45  # First infill angle (degrees)
INFILL_ANGLE_B = -45  # Alternate infill angle (degrees)

# Slicing performance settings
SLICE_TOLERANCE = 0.3  # mm - tolerance band for slicing (increase to speed up)
SEGMENT_MATCHING_TOLERANCE = 0.5  # mm - max distance to connect segments (increase to speed up)
ANGLE_MATCHING_TOLERANCE = 5.0  # degrees - max angle difference to match segments

# Path settings
SEGMENTS_PER_DEGREE = 2  # Resolution for circular paths

## Cylindrical Slicing Functions

In [ ]:
def slice_mesh_cylindrical(mesh: trimesh.Trimesh, radius: float, drum_radius: float, tolerance: float = 0.1) -> List[np.ndarray]:
    """
    Slice mesh at cylindrical surface by intersecting faces with the cylinder.
    Uses a tolerance band to handle inscribed polygon effect from triangulation.
    """
    segments = []
    
    # For each face, check if it intersects the cylinder
    for face in mesh.faces:
        v0, v1, v2 = mesh.vertices[face]
        r0 = np.sqrt(v0[1]**2 + v0[2]**2)
        r1 = np.sqrt(v1[1]**2 + v1[2]**2)
        r2 = np.sqrt(v2[1]**2 + v2[2]**2)
        
        vertices = [v0, v1, v2]
        radii = [r0, r1, r2]
        
        # Find edges that cross or are near the cylinder at this radius
        intersections = []
        for i in range(3):
            j = (i + 1) % 3
            ri, rj = radii[i], radii[j]
            r_min = min(ri, rj)
            r_max = max(ri, rj)
            
            if (r_min - tolerance <= radius <= r_max + tolerance):
                # Calculate intersection point
                if abs(ri - rj) > 0.001:
                    t = (radius - ri) / (rj - ri)
                    t = max(0.0, min(1.0, t))
                    p = vertices[i] + t * (vertices[j] - vertices[i])
                    
                    x = p[0]
                    theta = np.degrees(np.arctan2(p[2], p[1]))
                    intersections.append([x, theta])
        
        if len(intersections) == 2:
            segments.append(intersections)
    
    if not segments:
        return []
    
    # Filter out exact duplicates
    filtered_segments = []
    for seg in segments:
        is_duplicate = False
        for existing_seg in filtered_segments:
            if (points_close(seg[0], existing_seg[0], 0.01, 0.5) and 
                points_close(seg[1], existing_seg[1], 0.01, 0.5)) or \
               (points_close(seg[0], existing_seg[1], 0.01, 0.5) and 
                points_close(seg[1], existing_seg[0], 0.01, 0.5)):
                is_duplicate = True
                break
        
        if not is_duplicate:
            filtered_segments.append(seg)
    
    segments = filtered_segments
    
    if not segments:
        return []
    contours = []
    used = [False] * len(segments)
    
    for start_idx in range(len(segments)):
        if used[start_idx]:
            continue
        
        contour = [segments[start_idx][0], segments[start_idx][1]]
        used[start_idx] = True
        
        # extend contour
        max_iterations = len(segments) * 2
        for iteration in range(max_iterations):
            found = False
            best_match_idx = None
            best_match_dist = float('inf')
            best_match_end = True
            best_match_reverse = False
            
            for i in range(len(segments)):
                if used[i]:
                    continue
                
                seg = segments[i]
                threshold = SEGMENT_MATCHING_TOLERANCE
                
                # 1. Connect seg[0] to end of contour
                dist = point_distance(contour[-1], seg[0])
                if dist < best_match_dist and dist < threshold:
                    best_match_dist = dist
                    best_match_idx = i
                    best_match_end = True
                    best_match_reverse = False
                
                # 2. Connect seg[1] to end of contour (reversed segment)
                dist = point_distance(contour[-1], seg[1])
                if dist < best_match_dist and dist < threshold:
                    best_match_dist = dist
                    best_match_idx = i
                    best_match_end = True
                    best_match_reverse = True
                
                # 3. Connect seg[1] to start of contour
                dist = point_distance(contour[0], seg[1])
                if dist < best_match_dist and dist < threshold:
                    best_match_dist = dist
                    best_match_idx = i
                    best_match_end = False
                    best_match_reverse = False
                
                # 4. Connect seg[0] to start of contour (reversed segment)
                dist = point_distance(contour[0], seg[0])
                if dist < best_match_dist and dist < threshold:
                    best_match_dist = dist
                    best_match_idx = i
                    best_match_end = False
                    best_match_reverse = True
            
            if best_match_idx is not None:
                seg = segments[best_match_idx]
                used[best_match_idx] = True
                found = True
                
                if best_match_end:
                    if best_match_reverse:
                        contour.append(seg[0])
                    else:
                        contour.append(seg[1])
                else:
                    if best_match_reverse:
                        contour.insert(0, seg[1])
                    else:
                        contour.insert(0, seg[0])
            
            if not found:
                break
        
        # Check if contour is closed
        if len(contour) >= 4:
            dist = point_distance(contour[0], contour[-1])
            
            if dist < SEGMENT_MATCHING_TOLERANCE:
                contour.pop()
                contours.append(np.array(contour))
            else:
                print(f"Skipping unclosed contour with {len(contour)} points (gap: {dist:.3f}mm)")
    
    return contours


def point_distance(p1, p2):
    """Calculate distance between two points in (x, theta) space."""
    dx = abs(p1[0] - p2[0])
    dtheta = abs(p1[1] - p2[1])
    
    # Handle theta wrapping
    if dtheta > 180:
        dtheta = 360 - dtheta
    arc_dist = dtheta * 3.14159 / 180 * 28
    
    return np.sqrt(dx**2 + arc_dist**2)


def points_close(p1, p2, x_thresh=0.5, theta_thresh=5.0):
    """Check if two points are close in (x, theta) space."""
    dx = abs(p1[0] - p2[0])
    dtheta = abs(p1[1] - p2[1])
    
    # Handle theta wrapping
    if dtheta > 180:
        dtheta = 360 - dtheta
    
    return dx < x_thresh and dtheta < theta_thresh


def connect_contour_segments(segments: List[np.ndarray], angle_threshold: float = 5.0) -> List[List[np.ndarray]]:
    """Convert segments to list format."""
    if not segments or len(segments) == 0:
        return []
    return [seg.tolist() if isinstance(seg, np.ndarray) else seg for seg in segments]


def offset_contour_inward(contour: List[np.ndarray], offset_distance: float, drum_radius: float, z_height: float) -> List[np.ndarray]:
    """
    Offset contour inward using morphological erosion on unwrapped cylindrical surface.
    """
    if len(contour) < 3:
        return []
    
    try:
        contour_array = np.array(contour)
        current_radius = drum_radius + z_height
        
        theta_rad = np.radians(contour_array[:, 1])
        theta_unwrapped = np.unwrap(theta_rad)
        
        unwrapped = np.column_stack([
            contour_array[:, 0],
            current_radius * theta_unwrapped
        ])
        
        poly = Polygon(unwrapped)
        if not poly.is_valid:
            print(f"WARNING: Initial polygon invalid!")
            poly = poly.buffer(0)
            if not poly.is_valid:
                print("ERROR: Could not fix invalid polygon")
                return []
        
        # Ensure correct winding order
        if not poly.exterior.is_ccw:
            poly = Polygon(list(poly.exterior.coords)[::-1])
        
        if poly.is_empty or poly.area < 0.001:
            return []
        
        # Morphological erosion with round joins
        offset_poly = poly.buffer(-offset_distance, join_style=1, resolution=16)
        
        if offset_poly.is_empty or offset_poly.area < 0.001:
            print(f"WARNING: Offset polygon empty or too small (offset={offset_distance:.3f}mm)")
            return []
        
        # Extract coordinates from result
        if hasattr(offset_poly, 'exterior'):
            coords = np.array(list(offset_poly.exterior.coords[:-1]))
        elif hasattr(offset_poly, 'geoms'):
            if len(offset_poly.geoms) == 0:
                return []
            largest = max(offset_poly.geoms, key=lambda p: p.area if hasattr(p, 'area') else 0)
            if hasattr(largest, 'exterior'):
                coords = np.array(list(largest.exterior.coords[:-1]))
            else:
                return []
        else:
            return []
        
        if len(coords) < 3:
            print(f"WARNING: Offset result has only {len(coords)} points")
            return []
        
        result = coords.copy()
        result[:, 1] = np.degrees(coords[:, 1] / current_radius)
        result[:, 1] = ((result[:, 1] + 180) % 360) - 180
        return result.tolist()
        
    except Exception as e:
        print(f"ERROR in offset_contour_inward: {e}")
        import traceback
        traceback.print_exc()
        return []


def generate_walls_for_contour(contour: List[np.ndarray], num_walls: int, wall_spacing: float, drum_radius: float, z_height: float) -> List[List[np.ndarray]]:
    """Generate wall perimeters by morphological erosion."""
    walls = [contour]
    
    for wall_idx in range(1, num_walls):
        offset_distance = wall_idx * wall_spacing
        inner_contour = offset_contour_inward(contour, offset_distance, drum_radius, z_height)
        
        if len(inner_contour) > 0:
            walls.append(inner_contour)
        else:
            break
    
    return walls


def generate_rectilinear_infill(contour: List[np.ndarray], line_spacing: float, angle_degrees: float, radius: float) -> List[List[np.ndarray]]:
    """Generate rectilinear infill in unwrapped coordinates."""
    if len(contour) < 3:
        return []
    
    try:
        from shapely.geometry import Polygon, LineString
        from shapely import affinity
        
        contour_array = np.array(contour)
        theta_rad = np.radians(contour_array[:, 1])
        theta_unwrapped = np.unwrap(theta_rad)
        
        unwrapped = np.column_stack([
            contour_array[:, 0],
            radius * theta_unwrapped
        ])
        
        poly = Polygon(unwrapped)
        
        if not poly.is_valid:
            poly = poly.buffer(0)
        
        if not poly.is_valid or poly.is_empty or poly.area < 0.001:
            return []
        
        minx, miny, maxx, maxy = poly.bounds
        infill_lines = []
        diagonal = np.sqrt((maxx - minx)**2 + (maxy - miny)**2)
        
        y = miny - diagonal
        while y <= maxy + diagonal:
            line = LineString([(minx - diagonal, y), (maxx + diagonal, y)])
            center_x = (minx + maxx) / 2
            center_y = (miny + maxy) / 2
            rotated_line = affinity.rotate(line, angle_degrees, origin=(center_x, center_y))
            
            intersection = poly.intersection(rotated_line)
            
            if not intersection.is_empty:
                if intersection.geom_type == 'LineString':
                    coords = np.array(list(intersection.coords))
                    coords[:, 1] = np.degrees(coords[:, 1] / radius)
                    coords[:, 1] = ((coords[:, 1] + 180) % 360) - 180
                    infill_lines.append(coords.tolist())
                elif intersection.geom_type == 'MultiLineString':
                    for line_seg in intersection.geoms:
                        coords = np.array(list(line_seg.coords))
                        coords[:, 1] = np.degrees(coords[:, 1] / radius)
                        coords[:, 1] = ((coords[:, 1] + 180) % 360) - 180
                        infill_lines.append(coords.tolist())
            
            y += line_spacing
        
        return infill_lines
        
    except Exception as e:
        return []

## Load and Prepare 3D Model

Insert the path to your 3D model

In [ ]:
# Load your STL filea
test_mesh = trimesh.load('Body3.stl')

print(f"Mesh loaded: {len(test_mesh.vertices)} vertices, {len(test_mesh.faces)} faces")
print(f"Mesh bounds - X: [{test_mesh.bounds[0][0]:.1f}, {test_mesh.bounds[1][0]:.1f}]")
print(f"             Y: [{test_mesh.bounds[0][1]:.1f}, {test_mesh.bounds[1][1]:.1f}]")
print(f"             Z: [{test_mesh.bounds[0][2]:.1f}, {test_mesh.bounds[1][2]:.1f}]")

# Calculate radial extent
all_radii = np.sqrt(test_mesh.vertices[:, 1]**2 + test_mesh.vertices[:, 2]**2)
min_radius = np.min(all_radii)
max_radius = np.max(all_radii)

print(f"Radial extent: {min_radius:.2f}mm to {max_radius:.2f}mm")

## Slice the Model

In [ ]:
# Slice the model
all_model_radii = np.sqrt(test_mesh.vertices[:, 1]**2 + test_mesh.vertices[:, 2]**2)
min_model_radius = np.min(all_model_radii)
max_model_radius = np.max(all_model_radii)

slice_start_radius = min_model_radius + LAYER_HEIGHT
slice_end_radius = max_model_radius
num_layers = int(np.ceil((slice_end_radius - slice_start_radius) / LAYER_HEIGHT))

all_slices = []

import time
start_time = time.time()

for layer_idx in range(num_layers):
    layer_start = time.time()
    
    actual_radius = slice_start_radius + layer_idx * LAYER_HEIGHT
    z_height = actual_radius - DRUM_RADIUS
    segments = slice_mesh_cylindrical(test_mesh, actual_radius, DRUM_RADIUS, tolerance=SLICE_TOLERANCE)
    
    if segments:
        contours = connect_contour_segments(segments)
        
        if contours:
            all_slices.append({
                'layer': layer_idx,
                'z_height': z_height,
                'radius': actual_radius,
                'contours': contours
            })
    
    layer_time = time.time() - layer_start
    if layer_idx % 5 == 0:
        print(f"Layer {layer_idx}/{num_layers}: {layer_time:.2f}s")

total_time = time.time() - start_time
print(f"\nSliced {num_layers} layers in {total_time:.1f}s (avg {total_time/num_layers:.2f}s/layer)")
print(f"Generated {len(all_slices)} slices")

## Visualize Slices (2D)

In [ ]:
# Plot some slices in 2D (unwrapped cylinder)
if len(all_slices) > 0:
    num_plots = min(6, len(all_slices))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    slice_indices = np.linspace(0, len(all_slices)-1, num_plots, dtype=int)
    
    for idx, slice_idx in enumerate(slice_indices):
        slice_data = all_slices[slice_idx]
        ax = axes[idx]
        
        for contour in slice_data['contours']:
            contour_array = np.array(contour)
            ax.plot(contour_array[:, 1], contour_array[:, 0], 'b-', linewidth=2)
        
        ax.set_xlabel('Angle (degrees)')
        ax.set_ylabel('X position (mm)')
        ax.set_title(f"Layer {slice_data['layer']}: Z={slice_data['z_height']:.2f}mm")
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-180, 180)
    
    plt.tight_layout()
    plt.show()
else:
    print("No slices to visualize")
    
# Visualize wall offsetting for a single layer
if len(all_slices) > 0:
    print("\\n=== Wall Offset Visualization ===")
    
    # Pick middle layer for visualization
    mid_layer_idx = len(all_slices) // 2
    slice_data = all_slices[mid_layer_idx]
    
    if len(slice_data['contours']) > 0:
        fig, ax = plt.subplots(1, 1, figsize=(12, 6))
        
        # Get first contour and generate walls
        contour = slice_data['contours'][0]
        walls = generate_walls_for_contour(
            contour, 
            NUM_WALLS, 
            WALL_SPACING,
            DRUM_RADIUS,
            slice_data['z_height']
        )
        
        # Plot each wall in different color
        colors = ['blue', 'red', 'green', 'orange', 'purple']
        for wall_idx, wall in enumerate(walls):
            wall_array = np.array(wall)
            color = colors[wall_idx % len(colors)]
            label = f"Wall {wall_idx + 1}" + (" (outer)" if wall_idx == 0 else " (inner)" if wall_idx == len(walls)-1 else "")
            ax.plot(wall_array[:, 1], wall_array[:, 0], color=color, linewidth=2, label=label)
        
        ax.set_xlabel('Angle (degrees)')
        ax.set_ylabel('X position (mm)')
        ax.set_title(f"Multi-Wall Offsetting - Layer {slice_data['layer']}: Z={slice_data['z_height']:.2f}mm\\n"
                    f"{len(walls)} walls with {WALL_SPACING:.3f}mm spacing")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
        
        print(f"Generated {len(walls)} walls for layer {slice_data['layer']}")
        print(f"Wall spacing: {WALL_SPACING:.3f}mm")
    else:
        print("No contours in middle layer to visualize")
else:
    print("No slices available for wall offset visualization")

## Generate Multi-Wall Toolpaths

In [ ]:
# Generate toolpaths for all slices with top/bottom infill
all_toolpaths = []
total_layers = len(all_slices)

for slice_idx, slice_data in enumerate(all_slices):
    layer_paths = []
    
    # Determine if this is a top or bottom layer
    is_bottom_layer = slice_idx < TOP_BOTTOM_LAYERS
    is_top_layer = slice_idx >= (total_layers - TOP_BOTTOM_LAYERS)
    needs_infill = is_bottom_layer or is_top_layer
    
    # Choose infill angle (alternate between layers)
    infill_angle = INFILL_ANGLE_A if slice_idx % 2 == 0 else INFILL_ANGLE_B
    
    for contour in slice_data['contours']:
        # Generate wall perimeters
        walls = generate_walls_for_contour(
            contour, 
            NUM_WALLS, 
            WALL_SPACING,
            DRUM_RADIUS,
            slice_data['z_height']
        )
        layer_paths.extend(walls)
        
        if needs_infill and len(walls) > 0:
            # Get the innermost wall to use as infill boundary
            infill_boundary_offset = NUM_WALLS * WALL_SPACING
            infill_boundary = offset_contour_inward(
                contour,
                infill_boundary_offset,
                DRUM_RADIUS,
                slice_data['z_height']
            )
            
            if len(infill_boundary) > 0:
                # Generate rectilinear infill with proper radius
                infill_lines = generate_rectilinear_infill(
                    infill_boundary,
                    INFILL_LINE_SPACING,
                    infill_angle,
                    slice_data['radius']
                )
                layer_paths.extend(infill_lines)
    
    all_toolpaths.append({
        'layer': slice_data['layer'],
        'z_height': slice_data['z_height'],
        'radius': slice_data['radius'],
        'paths': layer_paths,
        'has_infill': needs_infill,
        'infill_angle': infill_angle if needs_infill else None
    })

print(f"Generated toolpaths for {len(all_toolpaths)} layers")
total_paths = sum(len(tp['paths']) for tp in all_toolpaths)
print(f"Total paths: {total_paths}")

## Convert to FullControl Steps

In [ ]:
def find_nearest_path(current_point, remaining_paths):
    """
    Find the nearest unvisited path to the current position.
    
    Returns: (best_index, best_distance, reverse_path)
    """
    if not remaining_paths:
        return None, float('inf'), False
    
    best_idx = 0
    best_dist = float('inf')
    best_reverse = False
    
    for idx, path in enumerate(remaining_paths):
        if len(path) == 0:
            continue
        
        # Check distance to start of path
        dist_to_start = point_distance(current_point, path[0])
        if dist_to_start < best_dist:
            best_dist = dist_to_start
            best_idx = idx
            best_reverse = False
        
        # Check distance to end of path (would need to reverse)
        dist_to_end = point_distance(current_point, path[-1])
        if dist_to_end < best_dist:
            best_dist = dist_to_end
            best_idx = idx
            best_reverse = True
    
    return best_idx, best_dist, best_reverse


def create_arc_travel(start_x, start_theta, end_x, end_theta, radius, segments=16):
    """
    Create travel move along arc on drum surface.
    
    For travel moves on the same layer (same Z/radius):
    - If only X changes: straight line along X axis
    - If only theta changes: arc around drum at constant X
    - If both change: combined path (move in X while rotating)
    
    Args:
        start_x: Starting X position
        start_theta: Starting angle in degrees
        end_x: Ending X position  
        end_theta: Ending angle in degrees
        radius: Z height (constant during travel)
        segments: Number of intermediate points

    """
    if segments < 2:
        segments = 2
    
    travel_points = []
    for i in range(segments + 1):
        t = i / segments
        x = start_x + t * (end_x - start_x)
        theta = start_theta + t * (end_theta - start_theta)
        travel_points.append([x, theta])
    
    return travel_points


def toolpaths_to_gcode_steps(all_toolpaths: List[dict], drum_radius: float) -> List:
    """
    Convert toolpaths to FullControl steps with optimized travel moves.
    """
    steps = []
    
    # Start with extruder off
    steps.append(fc.Extruder(on=False))
    current_pos = [0, 0]
    current_continuous_angle = 0.0
    
    for layer_data in all_toolpaths:
        z_height = layer_data['z_height']
        paths = layer_data['paths']
        
        if len(paths) == 0:
            continue
        
        remaining_paths = list(paths)
        while remaining_paths:
            nearest_idx, travel_dist, reverse = find_nearest_path(current_pos, remaining_paths)
            
            if nearest_idx is None:
                break
            
            path = remaining_paths.pop(nearest_idx)
            if reverse:
                path = list(reversed(path))
            
            if len(path) > 0:
                # For the first point, find the best continuous angle representation
                start_x, start_theta = path[0][0], path[0][1]
                angle_candidates = [start_theta, start_theta + 360, start_theta - 360, 
                                  start_theta + 720, start_theta - 720]
                best_start_angle = min(angle_candidates, key=lambda a: abs(a - current_continuous_angle))
                angle_shift = best_start_angle - start_theta
                
                # TRAVEL MOVE: Create arc travel at constant radius
                angular_distance = abs(best_start_angle - current_continuous_angle)
                num_travel_segments = max(2, int(angular_distance / 10))  # 1 segment per 10 degrees
                
                # Create smooth travel along drum surface
                travel_points = create_arc_travel(
                    start_x=current_pos[0],
                    start_theta=current_continuous_angle,
                    end_x=start_x,
                    end_theta=best_start_angle,
                    radius=z_height,
                    segments=num_travel_segments
                )
                
                steps.append(fc.Extruder(on=False))
                for travel_pt in travel_points:
                    steps.append(cylindrical_to_point(
                        x=travel_pt[0],
                        theta_degrees=travel_pt[1],
                        radius=z_height
                    ))
                
                steps.append(fc.Extruder(on=True))
                for point in path[1:]:
                    point_x, point_theta = point[0], point[1]
                    continuous_theta = point_theta + angle_shift
                    
                    steps.append(cylindrical_to_point(
                        x=point_x,
                        theta_degrees=continuous_theta,
                        radius=z_height
                    ))
                
                last_x, last_theta = path[-1][0], path[-1][1]
                last_continuous_theta = last_theta + angle_shift
                
                current_pos = [last_x, last_theta]
                current_continuous_angle = last_continuous_theta
    
    return steps

gcode_steps = toolpaths_to_gcode_steps(all_toolpaths, DRUM_RADIUS)

print(f"Generated {len(gcode_steps)} G-code steps")

## Visualize 3D Toolpath

In [ ]:
# Visualize the toolpath in 3D
viz_steps = convert_steps_for_visualization(gcode_steps, drum_diameter=DRUM_DIAMETER)
plot_data = fc.transform(viz_steps, 'plot', fc.PlotControls(raw_data=True))
import plotly.graph_objects as go

fig = go.Figure()
total_print_points = 0
for path in plot_data.paths:
    if path.extruder.on:
        total_print_points += len(path.xvals)

current_point = 0
for path in plot_data.paths:
    if path.extruder.on:
        # Printing path - create continuous gradient from blue to cyan
        num_points = len(path.xvals)
        
        if total_print_points > 1 and num_points > 1:
            color_values = []
            for i in range(num_points):
                progress = (current_point + i) / (total_print_points - 1)
                color_values.append(progress)
            
            current_point += num_points
            
            fig.add_trace(go.Scatter3d(
                mode='lines',
                x=path.xvals,
                y=path.yvals,
                z=path.zvals,
                line=dict(
                    color=color_values,
                    colorscale=[[0, 'rgb(0,150,255)'], [1, 'rgb(0,255,255)']],
                    width=3,
                    showscale=False
                ),
                showlegend=False,
                hoverinfo='skip'
            ))
        else:
            fig.add_trace(go.Scatter3d(
                mode='lines',
                x=path.xvals,
                y=path.yvals,
                z=path.zvals,
                line=dict(color='rgb(0,150,255)', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    else:
        fig.add_trace(go.Scatter3d(
            mode='lines',
            x=path.xvals,
            y=path.yvals,
            z=path.zvals,
            line=dict(color='rgb(150,0,0)', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))

all_x = [val for path in plot_data.paths if len(path.xvals) > 0 for val in path.xvals]
all_y = [val for path in plot_data.paths if len(path.yvals) > 0 for val in path.yvals]
all_z = [val for path in plot_data.paths if len(path.zvals) > 0 for val in path.zvals]

# Calculate base ranges
x_range = max(all_x) - min(all_x) if all_x else 1
y_range = max(all_y) - min(all_y) if all_y else 1
z_range = max(all_z) - min(all_z) if all_z else 1

margin = 0.3
x_range_with_margin = x_range * (1 + margin)
y_range_with_margin = y_range * (1 + margin)
z_range_with_margin = z_range * (1 + margin)
max_yz_range = max(y_range_with_margin, z_range_with_margin)

x_center = (max(all_x) + min(all_x)) / 2
y_center = (max(all_y) + min(all_y)) / 2
z_center = (max(all_z) + min(all_z)) / 2
aspect_x = x_range_with_margin / max_yz_range if max_yz_range > 0 else 1
aspect_y = 1
aspect_z = 1

fig.update_layout(
    width=1200,
    height=720,
    scene=dict(
        aspectmode='manual',
        aspectratio=dict(x=aspect_x, y=aspect_y, z=aspect_z),
        xaxis=dict(
            title='X (mm)', 
            range=[x_center - x_range_with_margin/2, x_center + x_range_with_margin/2]
        ),
        yaxis=dict(
            title='Y (mm)', 
            range=[y_center - max_yz_range/2, y_center + max_yz_range/2]
        ),
        zaxis=dict(
            title='Z (mm)', 
            range=[z_center - max_yz_range/2, z_center + max_yz_range/2]
        ),
    ),
    template='plotly_dark',
    showlegend=False,
    title='Cylindrical Slicer - 3D Toolpath Visualization'
)

fig.show()

In [ ]:
# Generate G-code
gcode = fc.transform(gcode_steps, 'gcode', fc.GcodeControls(
    printer_name='cylindrical',
    save_as='sliced_model',
    initialization_data={
        'drum_radius': DRUM_RADIUS,
        'extrusion_width': EXTRUSION_WIDTH,
        'extrusion_height': LAYER_HEIGHT,
        'material_flow_percent': FLOW_PERCENT,
        'nozzle_temp': 210,
        'fan_percent': 100,
        'print_speed': 1000
    }
))

print("G-code generated successfully!")
print(f"Total lines: {len(gcode.split(chr(10)))}")
print("\nFirst 20 lines:")
print('\n'.join(gcode.split('\n')[:20]))